<a href="https://colab.research.google.com/github/abhinav77040/Flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abhinav77040/Flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## Finding 1

The paper reports that its model improves prediction accuracy.

### Methodology question

How was the target label created? Was it measured directly from user behavior or derived from another model? Knowing this helps evaluate whether the labels accurately represent the outcome being predicted.

---

## Finding 2

The paper reports strong validation results.

### Methodology question

Was the validation performed using grouped or time-aware splits? If the same client or time period appears in both training and testing, the reported performance may be optimistic.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [10]:
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

features = [
    "search_volume",
    "competition",
    "word_count",
    "char_count",
    "days_since_last_update",
    "ctr",
    "avg_position"
]

target = "clicks_90d"

model_df = df[features + [target, "client_id"]].dropna()

X = model_df[features]
y = model_df[target]
groups = model_df["client_id"]

gkf = GroupKFold(n_splits=5)

train_idx, test_idx = next(gkf.split(X, y, groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

model = RandomForestRegressor(
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

group_mae = mean_absolute_error(
    y_test,
    pred
)

print(group_mae)

26.34846076961519


The grouped split produced a different MAE than the random split. This provides a more realistic estimate because pages from the same client are not shared between training and testing.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## Leakage audit

I reviewed the input features used by the model.

Observed:

- Only historical features were used.
- No future clicks or future impressions were included.
- The target column was not used as an input feature.
- No manually assigned recommendation labels were used during training.

This reduces the risk of data leakage.

In [11]:
results = pd.DataFrame({
    "Actual": y_test,
    "Prediction": pred
})

results["Absolute Error"] = abs(
    results["Actual"] -
    results["Prediction"]
)

results.sort_values(
    "Absolute Error",
    ascending=False
).head(10)

,Actual,Prediction,Absolute Error
14178,1408,24.03,1383.97
12050,1324,247.54,1076.46
29400,910,22.27,887.73
13139,893,15.00,878.00
7481,815,7.00,808.00
13193,796,31.22,764.78
12511,704,3.99,700.01
6836,724,30.60,693.40
20054,981,320.24,660.76
26774,685,38.20,646.80


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

The model provides decision-support by identifying pages that may benefit from further review.

The observed performance suggests that historical SEO signals contain useful information, but the model does not guarantee future traffic improvements.

These findings are directional and should be combined with human judgment.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.